# ESI heater: read-only diagnostic

Run the **single code cell** in a **fresh 64-bit Windows Python kernel**, with `notebooks/` beside the installed `esi` plugin folder (or set `PLUGIN_DIR`). Set `COM_PORT` below (`16` means COM16). Only the Python standard library and the existing bundled runtime/DLL are needed.

Before running, close Explorer and all ESI vendor utilities. Explorer shutdown requests OFF, so this report describes the subsequent state, **not necessarily the earlier ON failure**. If shutdown is unconfirmed, follow the instrument's local safety procedure; do not use this notebook to recover an occupied port.

This probe reads controller/heater activation, interlock states and enabled interlocks, limits, target temperature, voltage/current/temperature monitoring, LED RGB, identity/version data and raw configuration bytes. The vendor's heater-power and output-voltage getters return **targets**, not proof of delivered power. Failed/non-finite readings are not replaced by zero or previous measurements. Raw results and statuses remain in the report. LED color, interlock bits and normal temperatures do not prove cooling-water flow or electrical safety.

**No activation, setpoint, limit, configuration, interlock, baud, purge or rescan command is sent. Active outputs are not shut down by this probe either.** It uses the low-level driver, not the plugin's initializing `connect()`. Unavailable modules are not enabled to obtain data. Close is attempted only after a successful Open and returned calls; a failed Open is not followed by Close on a possibly unowned DLL session. Close does not certify an output shutdown.

The report is saved automatically under `esi/logs/esi_heater_readonly/`; its full path is printed. **Send the JSON file, including a partial report if the probe fails.** A pending record is saved before each DLL call. On a timeout or interrupt, no further DLL call or Close is attempted, even after a late return. Do not rerun: make the instrument safe locally before restarting the kernel. A watchdog cannot cancel a native DLL call.


In [ ]:
# Run once with Explorer and all ESI utilities closed. No output command is sent.
COM_PORT = 16
PLUGIN_DIR = None  # Normally leave None; keep notebooks/ beside the esi/ plugin folder.

import ctypes
import hashlib
import importlib.util
import json
import math
import sys
import threading
import time
import uuid
from datetime import datetime, timezone
from pathlib import Path

# Same defaults as the plugin; these deadlines do NOT cancel a native DLL call.
OPEN_TIMEOUT_S = 5.0
READ_TIMEOUT_S = 3.0

# Only these getters, Open and Close are allowed. No high-level connect()/disconnect().
READ_FIELDS = {
    'get_sw_version': ('version',),
    'get_dev_type': ('type',),
    'get_enable': ('enabled',),
    'get_main_state': ('hex', 'name'),
    'get_device_state': ('hex', 'flags'),
    'get_interlock_state': ('hex', 'flags'),
    'get_interlock_enable': ('mask',),
    'get_inputs': ('front_input', 'rear_input', 'fan_input'),
    'get_power_monitors': ('ac_in_ok', 'dc_out_ok'),
    'get_voltage_state': ('hex', 'flags'),
    'get_temperature_state': ('hex', 'flags'),
    'get_product_id': ('product_id',),
    'get_product_no': ('product_no',),
    'get_fw_version': ('version',),
    'get_fw_date': ('date',),
    'get_hw_type': ('type',),
    'get_hw_version': ('version',),
    'get_config_values': ('max_config', 'data_size', 'name_size'),
    'get_current_config': ('bytes',),
    'get_module_dev_type': ('type',),
    'get_module_product_id': ('product_id',),
    'get_module_product_no': ('product_no',),
    'get_module_fw_version': ('version',),
    'get_module_hw_version': ('version',),
    'get_module_activation_state': ('enabled',),
    'get_module_state': ('word',),
    'get_module_led_data': ('red', 'green', 'blue'),
    'get_module_data_ready_flags': ('flags',),
    'get_heat_ctrl_ilock_state': ('mask',),
    'get_heat_ctrl_hw_limits': ('max_voltage_v', 'max_current_a', 'max_power_w', 'max_temperature_c'),
    'get_heat_ctrl_voltage_limit': ('voltage_v',),
    'get_heat_ctrl_current_limit': ('current_a',),
    'get_heat_ctrl_power_limit': ('power_w',),
    'get_heat_ctrl_heater_temperature': ('temperature_c',),
    # The vendor header labels these two as TARGETS, not measured power/voltage.
    'get_heat_ctrl_output_voltage': ('target_output_voltage_v',),
    'get_heat_ctrl_heater_power': ('target_heater_power_w',),
    'get_heat_ctrl_monitoring': ('valid', 'output_voltage_v', 'heater_voltage_v', 'current_a', 'temperature_c'),
    'get_heat_ctrl_housekeeping': ('valid', 'volt_3v3', 'temp_cpu_c', 'volt_5v', 'volt_24v', 'temp_psu_c'),
}


def utc_now():
    return datetime.now(timezone.utc).isoformat()


def json_safe(value):
    if isinstance(value, float) and not math.isfinite(value):
        return None
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (tuple, list)):
        return [json_safe(v) for v in value]
    return value


class PassiveProbe:
    def __init__(self, path, com, metadata=None):
        self.path = Path(path)
        self.lock = threading.RLock()
        self.stop = False
        self.inflight = None
        self.thread = None
        self.device = None  # Retain the DLL instance if closure is unconfirmed.
        self.reusable = True
        self.report = {
            'schema': 'esi-heater-readonly-v1', 'started_utc': utc_now(),
            'com_port': com, 'read_only': True, 'metadata': metadata or {},
            'port': 'not_opened', 'outcome': 'pending', 'reads': {}, 'calls': [],
            'requires_kernel_restart': False,
            'limitations': [
                'Explorer shutdown requests OFF: this is not the preceding ON-failure state.',
                'No setters, output commands, interlock overrides, baud changes, purges or rescans.',
                'Communication Close is not a shutdown or an electrical safety certificate.',
                'Readbacks are sequential; ADC freshness is not established. Reads may consume ready flags.',
                'Unavailable modules are not enabled to obtain data; failed reads are not measurements.',
                'Interlock bits and LED RGB do not identify wiring, water flow or a fault cause.',
                'A blocked DLL call is not cancelled. No further DLL call or Close follows a timeout.',
            ],
        }

    def save(self):
        with self.lock:
            self.path.parent.mkdir(parents=True, exist_ok=True)
            temporary = self.path.with_suffix('.json.tmp')
            temporary.write_text(json.dumps(json_safe(self.report), indent=2, allow_nan=False) + '\n', encoding='utf-8')
            temporary.replace(self.path)

    def halt(self, reason):
        with self.lock:
            self.stop = True
            self.reusable = False
            self.report.update(outcome='aborted', reason=reason, requires_kernel_restart=True)
            self.save()

    def call(self, name, *args, label=None):
        if name not in READ_FIELDS and name not in ('open_port', 'close_port'):
            raise ValueError(f'Forbidden non-read call: {name}')
        with self.lock:
            if self.stop:
                return None
            record = {'method': name, 'args': list(args), 'started_utc': utc_now(),
                      'state': 'running', 'status': None, 'data': None}
            self.report['calls'].append(record)
            if label:
                self.report['reads'][label] = record
            limit = OPEN_TIMEOUT_S if name in ('open_port', 'close_port') else READ_TIMEOUT_S
            started = time.monotonic()
            self.inflight = (record, started, limit)
            self.save()  # Persist the pending call BEFORE entering the native DLL.
        try:
            if self.stop:
                return None
            result = getattr(self.device, name)(*args)
        except AttributeError as exc:
            with self.lock:
                record.update(state='unavailable', error=str(exc))
        except BaseException as exc:
            with self.lock:
                record.update(state='exception', error=f'{type(exc).__name__}: {exc}')
                self.halt(f'{name} raised an exception; no further DLL calls.')
        else:
            status, values = ((0, [int(result)]) if name == 'get_sw_version'
                              else (int(result[0]), list(result[1:])) if isinstance(result, tuple)
                              else (int(result), []))
            with self.lock:
                record.update(state='returned', status=status, raw_values=json_safe(values))
                if name == 'get_sw_version':
                    record['status_origin'] = 'local getter has no vendor status code'
                else:
                    record['status_text'] = self.device.describe_error(status)
                fields = READ_FIELDS.get(name, ())
                if status == 0 and len(values) == len(fields):
                    data = dict(zip(fields, json_safe(values)))
                    record['values_finite'] = all(v is not None for v in data.values())
                    if 'valid' in data:
                        record['measurement_usable'] = bool(data['valid']) and record['values_finite']
                        if not record['measurement_usable']:
                            data = {k: v if k == 'valid' else None for k, v in data.items()}
                    record['data'] = data
                if name == 'close_port' and status == 0:
                    self.report['port'] = 'closed'
                    self.reusable = not self.stop
                # Returned transport failures are not retried or used for recovery.
                if status in (-7, -8, -9, -10, -11, -12):
                    self.halt(f'{name} returned transport error {status}; no further DLL calls.')
        finally:
            with self.lock:
                record['elapsed_s'] = time.monotonic() - started
                record['finished_utc'] = utc_now()
                if record['elapsed_s'] > limit and not self.stop:
                    self.halt(f'{name} exceeded its {limit:g} s deadline; no further DLL calls.')
                record['returned_after_stop'] = self.stop
                if self.stop:
                    record['data'] = None  # A late/aborted read is evidence, not a valid measurement.
                self.inflight = None
                self.save()
        return record.get('data') if not self.stop else None

    def worker(self, factory):
        attempted_open = False
        try:
            self.device = factory()
            self.call('get_sw_version', label='dll_version')
            if self.stop:
                return
            attempted_open = True
            self.reusable = False
            self.report['port'] = 'open_attempted'
            if self.call('open_port', self.report['com_port']) is None:
                if not self.stop:
                    self.halt('Open did not confirm port ownership. No Close is attempted; make the instrument safe locally before restarting the kernel.')
                return
            with self.lock:
                self.report['port'] = 'open'
            identity = self.call('get_dev_type', label='controller_type')
            if not identity or identity['type'] != 0x8ED6:
                with self.lock:
                    self.report['identity_note'] = 'ESI identity not verified; further reads skipped.'
                return
            for name in (
                'get_enable', 'get_main_state', 'get_device_state', 'get_interlock_enable',
                'get_interlock_state', 'get_inputs', 'get_power_monitors', 'get_voltage_state',
                'get_temperature_state', 'get_product_id', 'get_product_no', 'get_fw_version',
                'get_fw_date', 'get_hw_type', 'get_hw_version',
            ):
                self.call(name, label='controller/' + name.removeprefix('get_'))
                if self.stop:
                    return
            config_info = self.call('get_config_values', label='config_info')
            config_supported = bool(config_info and config_info['data_size'] == 53)
            if config_supported:
                self.call('get_current_config', label='config_before')
            else:
                with self.lock:
                    self.report['config_note'] = '53-byte config not verified; raw config read skipped.'
            identity = self.call('get_module_dev_type', 0, label='heater/type')
            if identity and identity['type'] == 0xDB1C:
                for name in (
                    'get_module_activation_state', 'get_module_state', 'get_module_led_data',
                    'get_module_product_id', 'get_module_product_no', 'get_module_fw_version',
                    'get_module_hw_version', 'get_module_data_ready_flags',
                ):
                    self.call(name, 0, label='heater/' + name.removeprefix('get_module_'))
                for name in (
                    'get_heat_ctrl_ilock_state', 'get_heat_ctrl_hw_limits',
                    'get_heat_ctrl_voltage_limit', 'get_heat_ctrl_current_limit',
                    'get_heat_ctrl_power_limit', 'get_heat_ctrl_heater_temperature',
                    'get_heat_ctrl_output_voltage', 'get_heat_ctrl_heater_power',
                    'get_heat_ctrl_monitoring', 'get_heat_ctrl_housekeeping',
                ):
                    self.call(name, label='heater/' + name.removeprefix('get_heat_ctrl_'))
            else:
                with self.lock:
                    self.report['heater_note'] = 'HEAT-CTRL identity unavailable/mismatched; module not enabled and heater reads skipped.'
            if config_supported:
                self.call('get_current_config', label='config_after')
            self.call('get_main_state', label='controller/main_state_end')
        except BaseException as exc:
            with self.lock:
                self.report['exception'] = f'{type(exc).__name__}: {exc}'
                if attempted_open:
                    self.halt('Probe interrupted; no further DLL calls.')
        finally:
            # This same worker owns ALL calls. Never close after a timeout/interrupt.
            if attempted_open and not self.stop:
                self.report['port'] = 'close_attempted'
                self.call('close_port')
                if self.report['port'] != 'closed':
                    self.halt('Port closure unconfirmed. Close is not replayed.')
            with self.lock:
                if not self.stop:
                    calls = self.report['calls']
                    partial = ('exception' in self.report or 'identity_note' in self.report
                               or 'heater_note' in self.report or 'config_note' in self.report
                               or any(c['status'] != 0 or c['data'] is None
                                      or c.get('values_finite') is False
                                      or c.get('measurement_usable') is False for c in calls))
                    self.report['outcome'] = 'partial' if partial else 'complete'
                self.report['finished_utc'] = utc_now()
                self.add_decoded_states()
                self.save()

    def add_decoded_states(self):
        def data(label):
            return self.report['reads'].get(label, {}).get('data') or {}
        self.report['decoded'] = {}
        definitions = (
            ('heater_state_bits', 'heater/state', 'word',
             {'temperature_control_active': 0x100, 'module_active': 0x4000, 'device_active': 0x8000}),
            ('interlocks_monitored', 'controller/interlock_enable', 'mask',
             {'heater_1': 1, 'heater_2': 2, 'controller_front': 4, 'controller_rear': 8}),
            ('heater_interlock_bits', 'heater/ilock_state', 'mask',
             {'interlock_1_current': 1, 'interlock_2_current': 2, 'interlock_1_last': 4, 'interlock_2_last': 8}),
        )
        for title, label, field, masks in definitions:
            raw = data(label).get(field)
            if raw is not None:
                self.report['decoded'][title] = {key: bool(raw & mask) for key, mask in masks.items()}
        before, after = data('config_before').get('bytes'), data('config_after').get('bytes')
        self.report['decoded']['config_bytes_equal'] = (before == after) if before is not None and after is not None else None
        monitor = self.report['reads'].get('heater/monitoring')
        temperature = data('heater/monitoring').get('temperature_c')
        maximum = data('heater/hw_limits').get('max_temperature_c')
        if monitor and temperature is not None and maximum is not None:
            in_range = 0.0 <= temperature <= maximum
            monitor['temperature_within_hardware_range'] = in_range
            if not in_range:
                monitor['measurement_usable'] = False
                monitor['data'] = {k: v if k == 'valid' else None for k, v in monitor['data'].items()}
                if self.report['outcome'] == 'complete':
                    self.report['outcome'] = 'partial'

    def run(self, factory):
        self.save()  # Refuse any native call if the report cannot be created.
        self.thread = threading.Thread(target=self.worker, args=(factory,), daemon=True)
        self.thread.start()
        try:
            while self.thread.is_alive():
                self.thread.join(0.05)
                with self.lock:
                    if self.inflight:
                        record, started, limit = self.inflight
                        if time.monotonic() - started > limit:
                            record['deadline_exceeded'] = True
                            self.halt(f"{record['method']} is still running after {limit:g} s. Do not rerun; make the instrument safe locally, then restart the kernel.")
                            break
        except KeyboardInterrupt:
            self.halt('User interrupt: native call may still be running. No automatic Close. Make the instrument safe locally before restarting the kernel.')
        print(f"Report: {self.path.resolve()}")
        print(f"Result: {self.report['outcome']}; communication: {self.report['port']}")
        for label in ('controller/main_state', 'controller/interlock_enable', 'controller/interlock_state',
                      'heater/activation_state', 'heater/state', 'heater/led_data', 'heater/ilock_state',
                      'heater/heater_temperature', 'heater/voltage_limit', 'heater/current_limit',
                      'heater/power_limit', 'heater/monitoring'):
            record = self.report['reads'].get(label)
            if record:
                print(f"{label}: {record['data'] if record['data'] is not None else 'unavailable (see raw status)'}")
        if self.report['requires_kernel_restart']:
            print('DO NOT RERUN. Port/DLL state is unconfirmed; no software shutdown was attempted.')
            print('Make the instrument safe locally before restarting the kernel. Send the JSON report.')
        else:
            print('Send this JSON report. No output command was sent; this is NOT a safe-to-touch verdict.')


def explorer_plugin_dirs():
    """ESIBD Explorer's configured plugin folder (Windows settings), then its default."""
    found = []
    try:
        import winreg
        with winreg.OpenKey(winreg.HKEY_CURRENT_USER, r'Software\ESIBD LAB\ESIBD Explorer\General') as key:
            value = winreg.QueryValueEx(key, 'Plugin path')[0]
        if isinstance(value, str) and value and not value.startswith('@'):
            found.append(Path(value).expanduser())
    except (ImportError, OSError):
        pass
    found.append(Path.home() / 'ESIBD Explorer' / 'plugins')
    return found


def main():
    previous = globals().get('_ESI_HEATER_READONLY_SESSION')
    if previous is not None and ((previous.thread and previous.thread.is_alive()) or not previous.reusable):
        raise RuntimeError('Previous DLL session is unfinished or unconfirmed. Do not reconnect; make the instrument safe locally before restarting the kernel.')
    if not sys.platform.startswith('win') or ctypes.sizeof(ctypes.c_void_p) != 8:
        raise RuntimeError('Use a 64-bit Windows Python kernel on the controller PC.')
    if isinstance(COM_PORT, bool) or not isinstance(COM_PORT, int) or not 1 <= COM_PORT <= 255:
        raise ValueError('COM_PORT must be an integer from 1 to 255, e.g. 16 for COM16.')
    candidates = ([Path(PLUGIN_DIR)] if PLUGIN_DIR is not None else
                  [Path.cwd().parent / 'esi', Path.cwd(), Path.cwd() / 'esi', Path.cwd() / 'plugins' / 'esi',
                   *[plugins / 'esi' for plugins in explorer_plugin_dirs()]])
    plugin = next((p.resolve() for p in candidates if (p / 'esi_plugin.py').is_file()), None)
    if plugin is None:
        raise FileNotFoundError('Keep notebooks/ beside the installed esi/ plugin folder, or set PLUGIN_DIR.')
    base = plugin / 'vendor/runtime/esi/esi_base.py'
    dll = plugin / 'vendor/runtime/esi/vendor/x64/COM-ESI-CTRL.dll'
    catalog = plugin / 'vendor/runtime/error_codes.json'
    for path in (base, dll, catalog):
        if not path.is_file():
            raise FileNotFoundError(f'Missing bundled file: {path}')
    dll_sha = hashlib.sha256(dll.read_bytes()).hexdigest()  # recorded in the report
    module_name = '_esibd_bundled_esi_heater_readonly_base'
    spec = importlib.util.spec_from_file_location(module_name, base)
    if spec is None or spec.loader is None:
        raise ModuleNotFoundError(f'Cannot load bundled ESIBase from {base}')
    module = importlib.util.module_from_spec(spec)
    sys.modules[module_name] = module
    spec.loader.exec_module(module)
    timestamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
    path = plugin / 'logs' / 'esi_heater_readonly' / f'heater_{timestamp}_{uuid.uuid4().hex[:8]}.json'
    metadata = {'dll_path': str(dll), 'dll_sha256': dll_sha,
                'base_sha256': hashlib.sha256(base.read_bytes()).hexdigest(),
                'python': sys.version, 'platform': sys.platform,
                'baud_rate': 'No SetBaudRate call; Open behavior is supplied by the vendor DLL.'}
    probe = PassiveProbe(path, COM_PORT, metadata)
    globals()['_ESI_HEATER_READONLY_SESSION'] = probe
    print('PASSIVE DIAGNOSTIC: no activation, setpoint, limit, interlock or baud change.')
    probe.run(lambda: module.ESIBase(com=COM_PORT, dll_path=dll, error_codes_path=catalog))


main()
